# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells” using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/intro/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, ensuring a standards-based description and access to the FAIR dataset.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review the available record sets in the dataset. Each record set, field, and column is referenced by their `@id` per the Croissant standard.

In [ ]:
# List available record sets (tables), their @ids and field ids

print('--- Available Record Sets ---')
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name} | @id: {rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None)}")
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for f in rs.fields:
            print(f"    - {f.name} | @id: {f.id if hasattr(f, 'id') else getattr(f, '@id', None)} | type: {f.data_type if hasattr(f, 'data_type') else getattr(f, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All access is performed using the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @ids
record_set_ids = [getattr(rs, 'id', getattr(rs, '@id', None)) for rs in record_sets]
print(f"Record set @ids detected: {record_set_ids}\n")

dataframes = {}

for record_set in record_sets:
    record_set_id = getattr(record_set, 'id', getattr(record_set, '@id', None))
    # Load all records for each RecordSet
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet '{record_set.name}' (@id: {record_set_id})")
        print("Columns:", df.columns.tolist())
        print(df.head())
    else:
        print(f"No records could be loaded from RecordSet '{record_set.name}' (@id: {record_set_id})\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This may include removing outliers, transforming data distributions, or grouping data by attributes to prepare it for further analysis.

In [ ]:
# Choose a RecordSet for EDA (replace with actual @id if there are multiple)
if len(dataframes) == 0:
    print("No DataFrames available for analysis. Check previous steps for errors.")
else:
    # Select the first RecordSet with data for demonstration
    target_record_set_id = list(dataframes.keys())[0]
    target_df = dataframes[target_record_set_id]
    print(f"Analyzing DataFrame from RecordSet @id: {target_record_set_id}")
    
    # Try to auto-detect a likely numeric field using dtype
    numeric_candidates = target_df.select_dtypes(include=["float", "int"]).columns.tolist()
    if not numeric_candidates:
        # Try to convert columns that look like numerics
        for col in target_df.columns:
            with pd.option_context('mode.use_inf_as_na', True):
                try:
                    target_df[col] = pd.to_numeric(target_df[col], errors='coerce')
                except Exception:
                    continue
        numeric_candidates = target_df.select_dtypes(include=["float", "int"]).columns.tolist()
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = target_df[numeric_field].mean() if pd.notnull(target_df[numeric_field].mean()) else 10
        filtered_df = target_df[target_df[numeric_field] > threshold]
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (N={len(filtered_df)}):")
        print(filtered_df.head())

        # Normalization
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a likely categorical/group field
        possible_group_fields = [c for c in target_df.columns if c != numeric_field and target_df[c].dtype==object]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            try:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
                print(grouped_df.head())
            except Exception as e:
                print(f"Grouping failed: {e}")
    else:
        print("No numeric columns found for EDA. Please check the DataFrame's columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show a distribution plot for the detected numeric field using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        # Top 10 groups only
        order = filtered_df[group_field].value_counts().index[:10]
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field, order=order)
        plt.title(f"'{numeric_field}' by '{group_field}' (Top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, examine, extract, and analyze mass spectrometry dataset records using the `mlcroissant` library. You can extend this analysis with further statistical summaries, machine learning pipelines, or custom data processing workflows depending on research needs.